# Entraînement du Modèle de Détection des Maladies des Plantes

Ce notebook entraîne un modèle de deep learning pour classifier les maladies des plantes en utilisant le transfer learning avec MobileNetV2.

## 1. Importation

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np

## 2. Définition des paramètres pour l'entraînement :

- **DATA_SET_PATH** : Chemin vers le dossier contenant les images
- **IMG_SIZE** : Résolution des images d'entrée (224x224 pour MobileNetV2)
- **BATCH_SIZE** : Nombre d'images par lot d'entraînement
- **EPOCHS** : Nombre de passages complets sur le dataset

In [ ]:
DATA_SET_PATH = "data/" 
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 15

## 3. Chargement et Préparation des Données

Charge les images du répertoire et les divise automatiquement en ensembles d'entraînement (80%) et validation (20%).
Les images sont redimensionnées et organisées en lots.

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_SET_PATH,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int' 
)


val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_SET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int'
)

## 4. Affichage des Classes

Récupère les noms des classes détectées et affiche le nombre total de catégories à classifier.

In [ ]:
class_names = train_ds.class_names
num_classes = len(class_names)
print(f"Classes trouvées : {class_names}")

## 5. Optimisation des Performances de Chargement

Utilise le caching et le prefetching pour accélérer le chargement des données pendant l'entraînement.

In [ ]:
# Optimisation pour la rapidité
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

## 6. Augmentation des Données

Définit les transformations aléatoires appliquées aux images pendant l'entraînement pour améliorer la robustesse du modèle :
- Retournement horizontal et vertical
- Rotation aléatoire
- Zoom aléatoire
- Variation du contraste

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

## 7. Modèle de Base (Transfer Learning)

Charge le modèle pré-entraîné **MobileNetV2** depuis ImageNet. 
Ce modèle contient déjà des connaissances générales sur la reconnaissance d'objets visuels.
Les poids sont gelés pour utiliser le transfer learning.

In [ ]:
# Import du modèle pré-entraîné MobileNetV2
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False, 
    weights='imagenet'
)

base_model.trainable = False # On ne touche pas aux poids de base de Google


## 8. Construction du Modèle Final

Assemble un pipeline complet :
1. Augmentation des données
2. Normalisation des pixels (-1 à 1)
3. Extraction des features via MobileNetV2
4. Couches de classification personnalisées
5. Compilation avec l'optimiseur Adam

In [ ]:
# Assemblage du modèle final
model = models.Sequential([
    # Entrée
    layers.Input(shape=(224, 224, 3)),
    
    # Étape 1 : Augmentation (uniquement active pendant le training)
    data_augmentation,
    
    # Étape 2 : Normalisation (Pixels 0-255 -> -1 à 1)
    layers.Rescaling(1./127.5, offset=-1),
    
    # Étape 3 : Le "corps" du CNN (MobileNetV2)
    base_model,
    
    # Étape 4 : La "tête" de classification (ton intelligence spécifique)
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2), # Évite le surapprentissage
    layers.Dense(num_classes, activation='softmax') # Sortie : 1 probabilité par maladie
])


model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 9. Entraînement du Modèle

Lance l'entraînement sur le nombre d'epochs défini.
Les performances sont évaluées à chaque epoch sur les données de validation.

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

## 10. Sauvegarde du Modèle

Exporte le modèle entraîné au format HDF5

In [ ]:
# Sauvegarder le modèle
model.save('model/plant_disease_model.h5')

## 11. Visualisation des Résultats

Affiche les courbes de précision et de perte pour évaluer la performance du modèle sur les données d'entraînement et validation.

In [ ]:
# Afficher les courbes de performance
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(acc, label='Précision Entraînement')
plt.plot(val_acc, label='Précision Validation')
plt.legend()
plt.title('Précision')

plt.subplot(1, 2, 2)
plt.plot(loss, label='Perte Entraînement')
plt.plot(val_loss, label='Perte Validation')
plt.legend()
plt.title('Perte (Loss)')
plt.show()